In [26]:
import pandas as pd
import baseline_simulator
from utils import *
from pathlib import Path

%load_ext autoreload
%autoreload 2

import datetime
import os

import warnings
warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
sessions_file = Path(__name__).resolve().parents[1] / "data" / "Sessions3.csv"
sessions_df = pd.read_csv(sessions_file)

sessions_df = sessions_df.sort_values(by="startChargeTime")

# Create output folder for this simulation
current_time = datetime.datetime.now()
time_str = current_time.strftime("%Y-%m-%d_%H-%M-%S")
folder_path = f"results/all_sch/{time_str}"
os.makedirs(folder_path, exist_ok=True)

# Create empty list which will contain the data for the summary DataFrame
summary_rows = []

for month in range(1, 2):
    test_df = sessions_df[
        (pd.to_datetime(sessions_df["connectTime"]).dt.year == 2023)
        & (pd.to_datetime(sessions_df["connectTime"]).dt.month == month)
    ]

    test_df = test_df[test_df["DurationHrs"] > 0.5]
    test_df = test_df[test_df["cumEnergy_Wh"] > 0]
    test_df["choice"] = "SCHEDULED"

    sim = baseline_simulator.BaselineSimulator(
        test_df,
        verbose=False,
    )

    power_profiles, prices, hourly_prices = sim.simulate()
    session_results = get_session_results(
        test_df, power_profiles, prices, sim.power_rate, sim.TOU, sim.delta_t
    )
    file_path = f"{folder_path}/{month}_2023_all_scheduled.csv"
    session_results.to_csv(file_path)

    agg_power_profile = aggregate_power_profiles(test_df, power_profiles, sim.delta_t)
    charging_revenue, TOU_cost = get_profit(
        test_df, power_profiles, prices, sim.delta_t, sim.TOU
    )

    demand_charge_kwh = round(max(agg_power_profile), 2)
    demand_charge_cents = round(sim.cost_dc * demand_charge_kwh, 2)
    total_profit = round(charging_revenue - TOU_cost - demand_charge_cents, 2)
    charging_revenue = round(charging_revenue, 2)
    TOU_cost = round(TOU_cost, 2)
    energy_delivered = round(sum(session_results['energy_delivered']), 2)

    print("------------------------------------------------------------")
    print("Month", month)
    print("Total Profit", total_profit)
    print("Charging Revenue", charging_revenue)
    print("TOU Cost", TOU_cost)
    print("Demand Charge Costs (cents)", demand_charge_cents)
    print("Peak Power", demand_charge_kwh)
    print("Energy Delivered", energy_delivered)

    row = [
        month,
        total_profit,
        charging_revenue,
        TOU_cost, 
        demand_charge_cents,
        demand_charge_kwh,
        energy_delivered,
        agg_power_profile
    ]
    summary_rows.append(row)

columns = [
    'Month',
    'Total Profit (cents)',
    'Charging Revenue (cents)',
    'TOU Cost (cents)',
    'Demand Charge (cents)',
    'Peak Power (kWh)',
    'Energy Delivered (kW)',
    'Aggregate Power Profile (kW)'
]

summary_df = pd.DataFrame(data=summary_rows, columns=columns)
file_path = f"{folder_path}/summary.csv"
summary_df.to_csv(file_path)

Optimizing sessions: 100%|██████████| 202/202 [00:59<00:00,  3.40it/s]


------------------------------------------------------------
Month 1
Total Profit 68259.68
Charging Revenue 337423.31
TOU Cost 253553.63
Demand Charge Costs (cents) 15610.0
Peak Power 31.22
Energy Delivered 3192.2
